In [1]:
# show → code input visible by default
# hide-output → output hidden by default
# show hide-output → both (can combine on one line)

import pandas as pd
#from google.cloud import bigquery
from common_lib.sql import BigQueryConnector
from common_lib.export import export_notebook_html
import datetime as dt
import plotly.express as px


In [2]:
query_location = './sql/ltv.sql'
parameters = {
    'start_date':'2026-01-01',
    'end_date': dt.datetime.now().strftime('%Y-%m-%d'),
    'dayx':28
}

bqc = BigQueryConnector()
cost_info = bqc.print_cost_estimate(query=query_location, is_path=True, query_parameters=parameters)

This query will process 49.72 GB when run.
Estimated query cost: $0.33


In [3]:
refresh_data = False

In [4]:
data = pd.DataFrame()

if refresh_data:
    data = bqc.get(query='./sql/ltv.sql', is_path=True, query_parameters=parameters)
    data.to_pickle('./data/ltv_data.pkl')



In [5]:
data = pd.read_pickle('./data/ltv_data.pkl')

In [6]:
data.sort_values(by=['user_id', 'dt'], inplace=True)
data

,user_id,install_dt,dt,days_since_install,usd_iap_revenue_cumu,usd_iap_revenue_exc_seasons_cumu,usd_net_iap_revenue_cumu,usd_net_iap_revenue_exc_seasons_cumu
8560965,1000355231AF02C9,2026-02-18,2026-02-18,0,0.0,0.0,0.0,0.0
9009177,1000355231AF02C9,2026-02-18,2026-02-19,1,0.0,0.0,0.0,0.0
8388650,1000355231AF02C9,2026-02-18,2026-02-20,2,0.0,0.0,0.0,0.0
1162404,1000355231AF02C9,2026-02-18,2026-02-21,3,0.0,0.0,0.0,0.0
7363466,1000355231AF02C9,2026-02-18,2026-02-22,4,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...
6669397,FFFFF67B4ED7017A,2026-04-17,2026-05-11,24,0.0,0.0,0.0,0.0
7962812,FFFFF67B4ED7017A,2026-04-17,2026-05-12,25,0.0,0.0,0.0,0.0
2756575,FFFFF67B4ED7017A,2026-04-17,2026-05-13,26,0.0,0.0,0.0,0.0
2127910,FFFFF67B4ED7017A,2026-04-17,2026-05-14,27,0.0,0.0,0.0,0.0


In [7]:
data['install_dt'] = pd.to_datetime(data['install_dt'])
#data['install_cohort'] = data['install_dt'].dt.to_period('W').apply(lambda r: r.start_time)
data['install_cohort'] = data['install_dt'].dt.to_period('W').dt.start_time

ltv_cohorted = data.groupby(['install_cohort', 'days_since_install']).agg(
    usd_iap_revenue_cumu=('usd_iap_revenue_cumu', 'mean'),
    usd_iap_revenue_exc_seasons_cumu=('usd_iap_revenue_exc_seasons_cumu', 'mean'),
    usd_net_iap_revenue_cumu=('usd_net_iap_revenue_cumu', 'mean'),
    usd_net_iap_revenue_exc_seasons_cumu=('usd_net_iap_revenue_exc_seasons_cumu', 'mean'),
    users=('user_id', pd.Series.nunique)
).reset_index()

ltv_cohorted

,install_cohort,days_since_install,usd_iap_revenue_cumu,usd_iap_revenue_exc_seasons_cumu,usd_net_iap_revenue_cumu,usd_net_iap_revenue_exc_seasons_cumu,users
0,2025-12-29,0,0.209359,0.189062,0.146551,0.132343,10928
1,2025-12-29,1,0.327065,0.281088,0.228945,0.196761,10928
2,2025-12-29,2,0.452596,0.395522,0.316817,0.276865,10928
3,2025-12-29,3,0.507626,0.441262,0.355339,0.308883,10928
4,2025-12-29,4,0.573835,0.499699,0.401685,0.349789,10928
...,...,...,...,...,...,...,...
720,2026-06-15,24,1.515314,1.291680,1.098749,0.942205,10249
721,2026-06-15,25,1.548187,1.321952,1.124183,0.965819,10249
722,2026-06-15,26,1.578693,1.349354,1.146292,0.985755,10249
723,2026-06-15,27,1.611842,1.378824,1.170446,1.007333,10249


In [24]:
fig = px.line(ltv_cohorted, 
              x='days_since_install', 
              labels={'days_since_install': 'Days since install'},
              y='usd_net_iap_revenue_cumu',
              color='install_cohort',
              title='Players level distribution',
              width=1500,
              height=600,
              hover_data={'days_since_install': True, 'usd_net_iap_revenue_cumu': True, 'users': True})

fig.show()

In [9]:
data['install_dt'] = pd.to_datetime(data['install_dt'])
data['install_cohort'] = data['install_dt'].dt.to_period('D').dt.start_time

ltv_cohorted = data.groupby(['install_cohort', 'days_since_install']).agg(
    usd_iap_revenue_cumu=('usd_iap_revenue_cumu', 'mean'),
    usd_iap_revenue_exc_seasons_cumu=('usd_iap_revenue_exc_seasons_cumu', 'mean'),
    usd_net_iap_revenue_cumu=('usd_net_iap_revenue_cumu', 'mean'),
    usd_net_iap_revenue_exc_seasons_cumu=('usd_net_iap_revenue_exc_seasons_cumu', 'mean'),
    users=('user_id', 'count')
).reset_index()


In [10]:
fig = px.line(ltv_cohorted[ltv_cohorted['days_since_install'].isin([1, 3, 7, 14, 21, 28])], 
              x='install_cohort', 
              labels={'install_cohort': 'Install Cohort'},
              y='usd_net_iap_revenue_cumu',
              color='days_since_install',
              title='Players level distribution',
              width=1500,
              height=600,
              hover_data={'days_since_install': True, 'usd_net_iap_revenue_cumu': True, 'users': True})

fig.show()

In [16]:
# Calculate multipliers between key days
key_days = [1, 3, 7, 14, 28]

ltv_cohorted_multipliers = ltv_cohorted[['install_cohort']].drop_duplicates().copy()

for i in range(len(key_days) - 1):
    day1 = key_days[i]
    day2 = key_days[i + 1]
    col_name = f'multiplier_d{day1}_to_d{day2}'
    
    d1_data = ltv_cohorted[ltv_cohorted['days_since_install'] == day1].set_index('install_cohort')['usd_net_iap_revenue_cumu']
    d2_data = ltv_cohorted[ltv_cohorted['days_since_install'] == day2].set_index('install_cohort')['usd_net_iap_revenue_cumu']
    
    ltv_cohorted_multipliers[col_name] = ltv_cohorted['install_cohort'].map(d2_data / d1_data)

ltv_cohorted_multipliers = ltv_cohorted_multipliers.melt(id_vars=['install_cohort'], var_name='multiplier_range', value_name='multiplier').sort_values(by=['install_cohort', 'multiplier_range'])

ltv_cohorted_multipliers

,install_cohort,multiplier_range,multiplier
516,2026-01-01,multiplier_d14_to_d28,1.551303
0,2026-01-01,multiplier_d1_to_d3,1.583988
172,2026-01-01,multiplier_d3_to_d7,1.449203
344,2026-01-01,multiplier_d7_to_d14,1.454227
517,2026-01-02,multiplier_d14_to_d28,1.410049
...,...,...,...
514,2026-06-20,multiplier_d7_to_d14,1.731371
687,2026-06-21,multiplier_d14_to_d28,1.205780
171,2026-06-21,multiplier_d1_to_d3,1.310053
343,2026-06-21,multiplier_d3_to_d7,1.609575


In [17]:
fig = px.line(ltv_cohorted_multipliers, 
              x='install_cohort', 
              labels={'install_cohort': 'Install Cohort'},
              y='multiplier',
              color='multiplier_range',
              title='Players level distribution',
              width=1500,
              height=600,
              hover_data={'multiplier_range': True, 'multiplier': True})

fig.show()